In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [4]:
import os

for file in os.listdir("/kaggle/input/competitions/titanic"):
    print(file)

train.csv
test.csv
gender_submission.csv


In [5]:
train.head()
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
test["Survived"] = np.nan
data = pd.concat([train, test], ignore_index=True)

In [7]:
data["Title"] = data["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

data["Title"] = data["Title"].replace(
    ["Mlle","Ms","Lady","Mme","Countess","Dr","Major","Col","Sir","Jonkheer","Don","Rev","Capt"],
    "Rare"
)

In [8]:
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1

In [9]:
data["Cabin"] = data["Cabin"].fillna("U")
data["Cabin"] = data["Cabin"].str[0]

In [10]:
data["Age"] = data["Age"].fillna(data["Age"].median())
data["Fare"] = data["Fare"].fillna(data["Fare"].median())
data["Embarked"] = data["Embarked"].fillna(data["Embarked"].mode()[0])

In [11]:
data["TicketLength"] = data["Ticket"].apply(lambda x: len(str(x)))

In [12]:
data = pd.get_dummies(
    data,
    columns=["Sex","Embarked","Title","Cabin"],
    drop_first=True
)

In [13]:
train_clean = data[data["Survived"].notna()]
test_clean = data[data["Survived"].isna()]

In [14]:
X = train_clean.drop(["Survived","Name","Ticket","PassengerId"], axis=1)
y = train_clean["Survived"]

X_test = test_clean.drop(["Survived","Name","Ticket","PassengerId"], axis=1)

In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=7,
    random_state=42
)

model.fit(X, y)

RandomForestClassifier(max_depth=7, n_estimators=400, random_state=42)

In [16]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print("Accuracy:", scores.mean())

Accuracy: 0.827154604230745


In [17]:
predictions = model.predict(X_test)

In [18]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)